In [ ]:
import pyemma
import pyemma.coordinates as coor
import pyemma.msm as msm
import mdtraj as md
import numpy as np
import matplotlib.pyplot as plt
import pyemma.plots as mplt
from pyemma.coordinates import load
import deeptime as dt
from deeptime.decomposition import TICA
from pyemma.coordinates.data.fragmented_trajectory_reader import DataSource
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
files = ["...xtc file path"]

In [ ]:
len(files)

In [ ]:
# Load the coordinates
feat = coor.featurizer("protein.pdb")
feat.add_backbone_torsions()

In [ ]:
data = coor.load(files, features=feat)

In [ ]:
data[0].shape

In [ ]:
tica = TICA(dim=2,lagtime=5)

In [ ]:
tica.fit(data)

In [ ]:
tica_data = tica.transform(data)

In [ ]:
tica_data.shape

In [ ]:
combined_tica_data = np.concatenate(tica_data, axis=0)

In [ ]:
trajectory_index = 2
tica_components = tica_data[trajectory_index]

plt.scatter(tica_components[:, 0], tica_components[:, 1])
plt.xlabel("TICA Component 1")
plt.ylabel("TICA Component 2")
plt.title(f"TICA Projection - Trajectory {trajectory_index + 1}")
plt.show()

In [ ]:
trajectory_index = 2
tica_components = tica_data[trajectory_index]

# Calculate the mean (center) of the TICA components
mean_center = np.mean(tica_components, axis=0)
distances = np.sqrt(np.sum((tica_components - mean_center) ** 2, axis=1))

distance_threshold = np.percentile(distances, 95)

filtered_indices = distances <= distance_threshold
filtered_tica_components = tica_components[filtered_indices]

plt.scatter(filtered_tica_components[:, 0], filtered_tica_components[:, 1])
plt.xlabel("TICA Component 1")
plt.ylabel("TICA Component 2")
plt.title(f"TICA Projection - Trajectory {trajectory_index + 1} (Filtered)")
plt.show()


In [ ]:
Y = combined_tica_data

In [ ]:
from tqdm.notebook import tqdm
from deeptime.clustering import KMeans

estimator = KMeans(
    n_clusters=100,
    init_strategy='uniform',
    max_iter=500,
    fixed_seed=13,
    n_jobs=8,
    progress=tqdm
)

In [ ]:
clustering = estimator.fit(Y).fetch_model()

In [ ]:
assignments = clustering.transform(Y)

In [ ]:
len(clustering.cluster_centers)

In [ ]:
# Calculate implied timescales
its = msm.timescales_msm(assignments, lags=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50], nits=9)

In [ ]:
# Plot
mplt.plot_implied_timescales(its, dt=0.2, units='ns')
ax = plt.gca()
Image_filename = 'timescale.png'
plt.savefig(Image_filename, dpi=1200)
plt.show()

In [ ]:
Y=tica_data

In [ ]:
Y.shape

In [ ]:
def plot_labels(ax=None):
  for i in range(0,len(clustering.cluster_centers)):
        plt.text(clustering.cluster_centers[i][0]+0.05, clustering.cluster_centers[i][1]+0.05, str(i + 1), fontsize=10, color='black')

In [ ]:
pyemma.plots.plot_free_energy(np.vstack(Y)[:, 0], np.vstack(Y)[:, 1])
Image_filename = 'energy1.png'
plt.savefig(Image_filename, dpi=1200)

In [ ]:
plt.figure()
pyemma.plots.plot_free_energy(np.vstack(Y)[:, 0], np.vstack(Y)[:, 1])
cc_x = clustering.cluster_centers[:, 0]
cc_y = clustering.cluster_centers[:, 1]
plt.plot(cc_x, cc_y, linewidth=0, marker='o', markersize=5, color='black')
Image_filename = 'energy2.png'
plt.savefig(Image_filename, dpi=1200)

In [ ]:
Y = combined_tica_data
n_clusters = 100
cluster = pyemma.coordinates.cluster_kmeans(Y,k=n_clusters, max_iter=1000)
dtrajs = cluster.dtrajs

In [ ]:
dtrajs

In [ ]:
labels = np.concatenate(cluster.dtrajs)
mapping = defaultdict(lambda : [])
for i, label in enumerate(labels):
    mapping[label].append(i)
for i in range(0,len(clustering.cluster_centers)):
    print('size of cluster %d: %d structures' % (i+1, len(mapping[i])))

In [ ]:
msm = pyemma.msm.estimate_markov_model(cluster.dtrajs, lag=10)
print('fraction of states used = {:f}'.format(msm.active_state_fraction))
print('fraction of counts used = {:f}'.format(msm.active_count_fraction))

In [ ]:
cktest = msm.cktest(5, mlags=[0,1,2])
mplt.plot_cktest(cktest, dt=0.5, units='ns')
Image_filename = 'ck_plot.png'
plt.savefig(Image_filename, dpi=1200)
plt.show()

In [ ]:
dtrajs_concatenated = np.concatenate(cluster.dtrajs)

In [ ]:
fig, ax, misc = pyemma.plots.plot_contour(
    *combined_tica_data.T, msm.pi[dtrajs_concatenated],
    cbar_label='stationary_distribution',
    method='nearest', mask=True)


scatter = ax.scatter(combined_tica_data[:, 0], combined_tica_data[:, 1],
                     c=msm.pi[dtrajs_concatenated], s=15)

ax.set_xlabel('IC 1')
ax.set_ylabel('IC 2')
fig.tight_layout()

In [ ]:
nstates = 4
msm.pcca(nstates)

for i, s in enumerate(msm.metastable_sets):
    print('π_{} = {:f}'.format(i + 1, msm.pi[s].sum()))

fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for i, ax in enumerate(axes.flat):
    pyemma.plots.plot_contour(
        *combined_tica_data.T,
        msm.metastable_distributions[i][dtrajs_concatenated],
        ax=ax,
        cmap='afmhot_r',
        mask=True,
        cbar_label='metastable distribution {}'.format(i + 1))
    ax.set_xlabel('IC 1')
axes[0].set_ylabel('IC 2')
fig.tight_layout()

In [ ]:
metastable_traj = msm.metastable_assignments[dtrajs_concatenated]
highest_membership = msm.metastable_distributions.argmax(1)
coarse_state_centers = cluster.clustercenters[msm.active_set[highest_membership]]

In [ ]:
mfpt = np.zeros((nstates, nstates))
for i in range(nstates):
    for j in range(nstates):
        mfpt[i, j] = msm.mfpt(
            msm.metastable_sets[i],
            msm.metastable_sets[j])

inverse_mfpt = np.zeros_like(mfpt)
nz = mfpt.nonzero()
inverse_mfpt[nz] = 1.0 / mfpt[nz]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
_, _, misc = pyemma.plots.plot_state_map(
    *combined_tica_data.T, metastable_traj, ax=ax, zorder=-1)
misc['cbar'].set_ticklabels(range(1, nstates + 1))
ax.set_xlabel('IC 1')
ax.set_ylabel('IC 2')
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
fig.tight_layout()
ax.tick_params(axis='x',labelsize=16)
ax.tick_params(axis='y',labelsize=16)
ax.set_xlabel('IC1',fontsize=16)
ax.set_ylabel('IC2',fontsize=16)
misc['cbar'].ax.tick_params(labelsize=16)
misc['cbar'].set_label('state',fontsize=16)
image_filename = 'metastable.png'
plt.savefig(image_filename, dpi=1200)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

pyemma.plots.plot_network(
    inverse_mfpt,
    pos=coarse_state_centers,
    figpadding=0,
    arrow_label_format='%.1f ps',
    arrow_labels=mfpt,
    size=10,
    show_frame=True,
    ax=ax)

ax.set_xlabel('$\Phi$')
ax.set_ylabel('$\Psi$')
ax.set_xlim(-np.pi, np.pi)
ax.set_ylim(-np.pi, np.pi)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

num_states = len(inverse_mfpt)


node_labels = [str(i + 1) for i in range(num_states)]

pyemma.plots.plot_network(
    inverse_mfpt,
    pos=coarse_state_centers,
    figpadding=0,
    arrow_label_format='%.1f ps',
    arrow_labels=mfpt,
    size=10,
    show_frame=True,
    state_labels=node_labels,
    ax=ax
)

ax.set_xlabel('$\Phi$')
ax.set_ylabel('$\Psi$')
ax.set_xlim(-np.pi, np.pi)
ax.set_ylim(-np.pi, np.pi)
fig.tight_layout()


image_filename = 'flux.png'
plt.savefig(image_filename, dpi=1200)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for i, s in enumerate(msm.metastable_sets):
    mask = np.isin(dtrajs_concatenated, s)
    ax.scatter(
        combined_tica_data[mask, 0],  # IC 1
        combined_tica_data[mask, 1],  # IC 2
        label='S {}'.format(i + 1),
        alpha=0.7
    )

ax.set_xlabel('IC 1')
ax.set_ylabel('IC 2')
ax.set_title('Metastable Distributions')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
def euclidean_distance(point, centroid):
    return np.sqrt(np.sum((point - centroid)**2))

distance_threshold = 1.1

fig, ax = plt.subplots(figsize=(8, 6))

for i, s in enumerate(msm.metastable_sets):
    mask = np.isin(dtrajs_concatenated, s)
    metastate_data = combined_tica_data[mask]
    centroid = np.mean(metastate_data, axis=0)
    filtered_data = metastate_data[
        np.array([euclidean_distance(point, centroid) for point in metastate_data]) <= distance_threshold
    ]

    ax.scatter(
        filtered_data[:, 0],  # IC 1
        filtered_data[:, 1],
        marker='o',# IC 2
        label='S {}'.format(i + 1),
        alpha=0.7
    )

ax.set_xlabel('IC 1')
ax.set_ylabel('IC 2')
ax.set_title('Metastable Distributions')
ax.legend()
fig.tight_layout()
image_filename = 'metastable_filtered.png'
plt.savefig(image_filename, dpi=600)

plt.show()

In [ ]:
representative_structures = []

for i, structure in enumerate(representative_structures):
    traj = md.Trajectory([structure],model.pdb)
    traj.save_pdb(f"metastable_{i+1}.pdb")

In [ ]:
pos=np.asarray([[0, 0], [4, 0], [2, 4], [6, 4]])

In [ ]:
msm = pyemma.msm.bayesian_markov_model(cluster.dtrajs, lag=20, conf=0.95)
sample_mean = msm.sample_mean('timescales', k=1)
sample_conf_l, sample_conf_r = msm.sample_conf('timescales', k=1)

print('Mean of first ITS: {:f}'.format(sample_mean[0]))
print('Confidence interval: [{:f}, {:f}]'.format(sample_conf_l[0], sample_conf_r[0]))

In [ ]:
def its_separation_err(ts, ts_err):
    """
    Error propagation from ITS standard deviation to timescale separation.
    """
    return ts[:-1] / ts[1:] * np.sqrt(
        (ts_err[:-1] / ts[:-1])**2 + (ts_err[1:] / ts[1:])**2)


In [ ]:
nits = 15

timescales_mean = msm.sample_mean('timescales', k=nits)
timescales_std = msm.sample_std('timescales', k=nits)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].errorbar(
    range(1, nits + 1),
    timescales_mean,
    yerr=timescales_std,
    fmt='.', markersize=10)
axes[1].errorbar(
    range(1, nits),
    timescales_mean[:-1] / timescales_mean[1:],
    yerr=its_separation_err(
        timescales_mean,
        timescales_std),
    fmt='.',
    markersize=10,
    color='C0')

for i, ax in enumerate(axes):
    ax.set_xticks(range(1, nits + 1))
    ax.grid(True, axis='x', linestyle=':')

axes[0].axhline(msm.lag * 0.1, lw=1.5, color='k')
axes[0].axhspan(0, msm.lag * 0.1, alpha=0.3, color='k')
axes[0].set_xlabel('implied timescale index')
axes[0].set_ylabel('implied timescales / ns')


axes[1].set_xticks(range(1, nits))
axes[1].set_xticklabels(
    ["{:d}/{:d}".format(k, k + 1) for k in range(1, nits)],
    rotation=45)
axes[1].set_xlabel('implied timescale indices')
axes[1].set_ylabel('timescale separation')
fig.tight_layout()
Image_filename = 'index.png'
plt.savefig(Image_filename, dpi=1200)
